In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
import streamlit as st
from sklearn.tree import DecisionTreeClassifier

In [2]:
bball_df = pd.read_csv("combined_data1.csv")

In [3]:
new_bball = pd.DataFrame({
            "PlayResult" : bball_df["PlayResult"],
            "ExitSpeed" : bball_df["ExitSpeed"],
            "Angle": bball_df["Angle"],
            "Direction": bball_df["Direction"]
})

In [4]:
new_bball["PlayResult"] = np.where(new_bball["PlayResult"] == "Undefined",
                                   np.nan,
                                   new_bball["PlayResult"])

In [5]:

new_bball.isnull().sum()

PlayResult    6398
ExitSpeed     5742
Angle         5742
Direction     5742
dtype: int64

In [11]:

new_bball = new_bball.dropna()

In [13]:
new_bball.isnull().sum()

PlayResult    0
ExitSpeed     0
Angle         0
Direction     0
dtype: int64

In [15]:
new_bball = new_bball.copy()  # Ensure we are working with a copy
new_bball.loc[:, "Hit_or_Out"] = new_bball["PlayResult"].apply(lambda x: 1 if x in ["Single", "Double", "Triple", "Homerun"] else 0)

In [191]:
#target (y) and feature selection for the model(
y = new_bball["Hit_or_Out"]
X = new_bball[["ExitSpeed", "Angle"]]


In [193]:
#split into test and train set
X_train, X_test, y_train, y_test = train_test_split(X.values,
                                                    y,
                                                    stratify=y,       # same number of target in training & test set
                                                    test_size=0.2,    # hold out 20% of data for testing
                                                    random_state=206)

In [195]:
rf_classifier = RandomForestClassifier(n_estimators=30,
                                       max_depth = 7,
                                       min_samples_split = 3,
                                       random_state=206)

In [197]:
rf_classifier.fit(X_train, y_train)

RandomForestClassifier(max_depth=7, min_samples_split=3, n_estimators=30,
                       random_state=206)

See how model performs on the training set


In [236]:
y_pred_rf_train = clf.predict(X_train)

In [238]:
rf_con_train = confusion_matrix(y_train, y_pred_rf_train)
rf_con_train

array([[656, 116],
       [ 38, 270]], dtype=int64)

In [240]:
print(classification_report(y_train, y_pred_rf_train))

              precision    recall  f1-score   support

           0       0.95      0.85      0.89       772
           1       0.70      0.88      0.78       308

    accuracy                           0.86      1080
   macro avg       0.82      0.86      0.84      1080
weighted avg       0.88      0.86      0.86      1080



### Try it on the test set 

In [243]:
#predict the results on the test set
y_pred = rf_classifier.predict(X_test)

In [245]:
confusion_m = confusion_matrix(y_test, y_pred)

In [247]:
confusion_m

array([[175,  18],
       [ 37,  40]], dtype=int64)

In [249]:
pd.DataFrame(confusion_matrix(y_test, y_pred),
            columns=["Predicted out", "Predicted hit"],
            index=["Actual out","Actual Hit"]).style.background_gradient(cmap="PiYG")

,Predicted out,Predicted hit
Actual out,175,18
Actual Hit,37,40


In [251]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.91      0.86       193
           1       0.69      0.52      0.59        77

    accuracy                           0.80       270
   macro avg       0.76      0.71      0.73       270
weighted avg       0.79      0.80      0.79       270



### Try a decision tree

In [254]:
## try a decision tree
clf = DecisionTreeClassifier(criterion="gini", max_depth=7, random_state=206, class_weight = "balanced")
clf.fit(X_train, y_train)

DecisionTreeClassifier(class_weight='balanced', max_depth=7, random_state=206)

In [256]:
y_pred_dt_train = clf.predict(X_train)

In [258]:
dt_con_train = confusion_matrix(y_train, y_pred_dt_train)
dt_con_train

array([[656, 116],
       [ 38, 270]], dtype=int64)

In [260]:
print(classification_report(y_train, y_pred_dt_train))

              precision    recall  f1-score   support

           0       0.95      0.85      0.89       772
           1       0.70      0.88      0.78       308

    accuracy                           0.86      1080
   macro avg       0.82      0.86      0.84      1080
weighted avg       0.88      0.86      0.86      1080



In [262]:
y_pred_dt = clf.predict(X_test)

In [264]:
dt_confusion = confusion_matrix(y_test, y_pred_dt)

In [266]:
dt_confusion

array([[150,  43],
       [ 18,  59]], dtype=int64)

In [268]:
pd.DataFrame(confusion_matrix(y_test, y_pred_dt),
            columns=["Predicted out", "Predicted hit"],
            index=["Actual out","Actual Hit"]).style.background_gradient(cmap="PiYG")

,Predicted out,Predicted hit
Actual out,150,43
Actual Hit,18,59


In [270]:
print(classification_report(y_test, y_pred_dt))

              precision    recall  f1-score   support

           0       0.89      0.78      0.83       193
           1       0.58      0.77      0.66        77

    accuracy                           0.77       270
   macro avg       0.74      0.77      0.75       270
weighted avg       0.80      0.77      0.78       270

